In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, TimeSeriesSplit, cross_val_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, VotingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import joblib
import time
import warnings
warnings.filterwarnings('ignore')

In [6]:
# Load the feature-engineered dataset
print("Loading feature-engineered dataset...")
df = pd.read_csv('/workspace/Renewable-Energy-Prediction/data/processed/Feature_Engineering _Dataset/feature_engineered_data.csv', 
                 parse_dates=['time'])
print(f"Dataset shape: {df.shape}")


Loading feature-engineered dataset...
Dataset shape: (177210, 44)


In [7]:
# Check for missing values from the lag features
print("\nMissing values:")
print(df.isnull().sum().sum())


Missing values:
1464


In [8]:
# Fill missing values for lag features
print("\nFilling missing values...")
df = df.fillna(method='bfill')
missing_after = df.isnull().sum().sum()
print(f"Remaining missing values: {missing_after}")



Filling missing values...


Remaining missing values: 0


In [9]:
# Define feature columns
feature_cols = [col for col in df.columns if col not in ['Area', 'YEAR', 'Country', 'time', 'Solar', 'Wind Onshore']]
print(f"\nNumber of features: {len(feature_cols)}")


Number of features: 38


In [10]:
# Define evaluation function
def evaluate_model(y_true, y_pred, model_name, target_name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    
    print(f"{model_name} - {target_name} Metrics:")
    print(f"RMSE: {rmse:.2f}")
    print(f"MAE: {mae:.2f}")
    print(f"R²: {r2:.4f}")
    
    return {'rmse': rmse, 'mae': mae, 'r2': r2}

In [11]:
# Model training function for a specific target (Wind and Solar)
def train_evaluate_models(X_train, X_test, y_train, y_test, target_name):
    results = {}
    models = {}
    
    print(f"\n===== Training models for {target_name} prediction =====")
    
    # 1. Random Forest
    print("\nTraining Random Forest...")
    start_time = time.time()
    rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    rf_model.fit(X_train, y_train)
    rf_pred = rf_model.predict(X_test)
    rf_time = time.time() - start_time
    print(f"Training time: {rf_time:.2f} seconds")
    
    rf_results = evaluate_model(y_test, rf_pred, "Random Forest", target_name)
    results["Random Forest"] = rf_results
    models["Random Forest"] = rf_model
    
    # Feature importance for Random Forest
    rf_feature_imp = pd.DataFrame({
        'Feature': X_train.columns,
        'Importance': rf_model.feature_importances_
    }).sort_values('Importance', ascending=False).head(10)
    
    print("\nTop 10 features for Random Forest:")
    print(rf_feature_imp)
    
    # 2. Gradient Boosting Machine
    print("\nTraining GBM...")
    start_time = time.time()
    gbm_model = GradientBoostingRegressor(n_estimators=100, random_state=42)
    gbm_model.fit(X_train, y_train)
    gbm_pred = gbm_model.predict(X_test)
    gbm_time = time.time() - start_time
    print(f"Training time: {gbm_time:.2f} seconds")
    
    gbm_results = evaluate_model(y_test, gbm_pred, "GBM", target_name)
    results["GBM"] = gbm_results
    models["GBM"] = gbm_model
    
    # 3. XGBoost
    print("\nTraining XGBoost...")
    start_time = time.time()
    xgb_model = xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, n_jobs=-1)
    xgb_model.fit(X_train, y_train)
    xgb_pred = xgb_model.predict(X_test)
    xgb_time = time.time() - start_time
    print(f"Training time: {xgb_time:.2f} seconds")
    
    xgb_results = evaluate_model(y_test, xgb_pred, "XGBoost", target_name)
    results["XGBoost"] = xgb_results
    models["XGBoost"] = xgb_model
    
    # 4. Simple Ensemble (Voting Regressor)
    print("\nTraining Simple Ensemble (Voting Regressor)...")
    start_time = time.time()
    ensemble_model = VotingRegressor(
        estimators=[
            ('rf', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)),
            ('gbm', GradientBoostingRegressor(n_estimators=100, random_state=42)),
            ('xgb', xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, n_jobs=-1))
        ]
    )
    ensemble_model.fit(X_train, y_train)
    ensemble_pred = ensemble_model.predict(X_test)
    ensemble_time = time.time() - start_time
    print(f"Training time: {ensemble_time:.2f} seconds")
    
    ensemble_results = evaluate_model(y_test, ensemble_pred, "Ensemble", target_name)
    results["Ensemble"] = ensemble_results
    models["Ensemble"] = ensemble_model
    
    # Compare model performance
    performance_df = pd.DataFrame(results).T
    print("\nModel Performance Comparison:")
    print(performance_df)
       
    return results, models


In [12]:
# Main execution
print("\nPreparing data for modeling...")


Preparing data for modeling...


In [13]:
# Time-based split (more appropriate for time-series data)
# We'll use the last 20% of the data as test set to maintain time ordering
split_idx = int(len(df) * 0.8)
train_df = df.iloc[:split_idx]
test_df = df.iloc[split_idx:]

print(f"Training set size: {train_df.shape}")
print(f"Test set size: {test_df.shape}")

Training set size: (141768, 44)
Test set size: (35442, 44)


In [14]:
# Prepare features and targets
X_train = train_df[feature_cols]
y_train_solar = train_df['Solar']
y_train_wind = train_df['Wind Onshore']

X_test = test_df[feature_cols]
y_test_solar = test_df['Solar']
y_test_wind = test_df['Wind Onshore']

In [15]:
# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [16]:
# Convert back to DataFrame to maintain column names
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

In [17]:
# Save the scaler
joblib.dump(scaler, '/workspace/Renewable-Energy-Prediction/models/feature_scaler.pkl')

['/workspace/Renewable-Energy-Prediction/models/feature_scaler.pkl']

In [18]:
# Train and evaluate models for Solar
solar_results, solar_models = train_evaluate_models(X_train_scaled, X_test_scaled, y_train_solar, y_test_solar, "Solar")



===== Training models for Solar prediction =====

Training Random Forest...


Training time: 141.63 seconds
Random Forest - Solar Metrics:
RMSE: 400.04
MAE: 202.06
R²: 0.9853

Top 10 features for Random Forest:
                   Feature  Importance
22            Solar_lag_24    0.813821
19             Solar_lag_1    0.148074
30   Solar_rolling_24h_std    0.009902
6                     hour    0.007620
11                hour_sin    0.007131
21             Solar_lag_3    0.003225
23           Solar_lag_168    0.002090
12                hour_cos    0.002054
20             Solar_lag_2    0.001246
29  Solar_rolling_24h_mean    0.001077

Training GBM...
Training time: 92.79 seconds
GBM - Solar Metrics:
RMSE: 471.24
MAE: 266.77
R²: 0.9797

Training XGBoost...
Training time: 24.68 seconds
XGBoost - Solar Metrics:
RMSE: 484.27
MAE: 241.17
R²: 0.9785

Training Simple Ensemble (Voting Regressor)...
Training time: 255.68 seconds
Ensemble - Solar Metrics:
RMSE: 427.56
MAE: 223.46
R²: 0.9833

Model Performance Comparison:
                     rmse         mae        r2
Rando

In [20]:
# Train and evaluate models for Wind
wind_results, wind_models = train_evaluate_models(X_train_scaled, X_test_scaled, y_train_wind, y_test_wind, "Wind Onshore")



===== Training models for Wind Onshore prediction =====

Training Random Forest...
Training time: 135.02 seconds
Random Forest - Wind Onshore Metrics:
RMSE: 341.23
MAE: 233.89
R²: 0.9909

Top 10 features for Random Forest:
                          Feature  Importance
24             Wind Onshore_lag_1    0.989338
25             Wind Onshore_lag_2    0.004141
26             Wind Onshore_lag_3    0.001007
32   Wind Onshore_rolling_24h_std    0.000495
31  Wind Onshore_rolling_24h_mean    0.000362
5                            pres    0.000324
28           Wind Onshore_lag_168    0.000298
27            Wind Onshore_lag_24    0.000287
29         Solar_rolling_24h_mean    0.000261
2                            rhum    0.000242

Training GBM...
Training time: 93.36 seconds
GBM - Wind Onshore Metrics:
RMSE: 406.06
MAE: 285.65
R²: 0.9872

Training XGBoost...
Training time: 7.47 seconds
XGBoost - Wind Onshore Metrics:
RMSE: 478.51
MAE: 272.58
R²: 0.9822

Training Simple Ensemble (Voting Regressor

In [24]:
# Compare Solar vs Wind model performance across algorithms
solar_performance = pd.DataFrame(solar_results).T
wind_performance = pd.DataFrame(wind_results).T


In [25]:
solar_performance['Target'] = 'Solar'
wind_performance['Target'] = 'Wind'

In [26]:
print("\n===== Overall Model Performance Summary =====")
overall_performance = pd.concat([solar_performance, wind_performance])
print(overall_performance)


===== Overall Model Performance Summary =====
                     rmse         mae        r2 Target
Random Forest  400.043291  202.055154  0.985337  Solar
GBM            471.239852  266.773655  0.979653  Solar
XGBoost        484.270017  241.166908  0.978513  Solar
Ensemble       427.558250  223.462852  0.983251  Solar
Random Forest  341.229776  233.889362  0.990940   Wind
GBM            406.057167  285.646897  0.987170   Wind
XGBoost        478.510558  272.582020  0.982183   Wind
Ensemble       376.037508  249.351244  0.988997   Wind


In [27]:
# Determine best model for each target
best_solar_model = solar_performance.sort_values('rmse').index[0]
best_wind_model = wind_performance.sort_values('rmse').index[0]

print(f"\nBest model for Solar prediction: {best_solar_model}")
print(f"Best model for Wind prediction: {best_wind_model}")


Best model for Solar prediction: Random Forest
Best model for Wind prediction: Random Forest
